# combRUM Quickstart

This is the smallest useful public-API path: define a pricing oracle, define a feature map, wrap them in `Model` and `Data`, fit with `estimate`, and bootstrap with `bootstrap`.

For each observation $i$ and simulation draw $s$, combRUM asks the oracle to solve

$$d^*_{is}(\theta) \in \arg\max_{d \in \mathcal{D}_i} \phi_i(d)^\top \theta + \varepsilon_{is}(d).$$

The package owns row generation and the master LP. The application owns the combinatorial subproblem and the feature rows.

## Data

A small, well-conditioned bundle-choice panel: each item has three observed covariates, each agent chooses any subset of four items, and simulated shocks enter item utility additively.

In [1]:
import itertools

import numpy as np

import combrum as cb

rng = np.random.default_rng(17)

N_ITEMS = 4
N_FEATURES = 3
N_OBS = 100
N_SIMULATIONS = 4
THETA_TRUE = np.array([0.9, -0.5, 0.35], dtype=np.float64)
SHOCK_SCALE = 0.10

BUNDLES = np.array(
    list(itertools.product([0.0, 1.0], repeat=N_ITEMS)),
    dtype=np.float64,
)
X = rng.normal(size=(N_OBS, N_ITEMS, N_FEATURES))
estimation_shocks = rng.normal(
    scale=SHOCK_SCALE,
    size=(N_OBS, N_SIMULATIONS, N_ITEMS),
)
observed_shocks = rng.normal(scale=SHOCK_SCALE, size=(N_OBS, N_ITEMS))


def simulate_observed(theta):
    observed = np.zeros((N_OBS, N_ITEMS), dtype=np.float64)
    for i in range(N_OBS):
        utilities = X[i] @ theta + observed_shocks[i]
        scores = BUNDLES @ utilities
        observed[i] = BUNDLES[int(np.argmax(scores))]
    return observed


observed_bundles = simulate_observed(THETA_TRUE)

{
    "observed_bundles": observed_bundles.shape,
    "mean_bundle_size": float(observed_bundles.sum(axis=1).mean()),
    "choice_share_by_item": observed_bundles.mean(axis=0).round(3).tolist(),
}

{'observed_bundles': (100, 4),
 'mean_bundle_size': 2.04,
 'choice_share_by_item': [0.49, 0.57, 0.48, 0.5]}

## Oracle And Feature Map

`BundleOracle` is the model-specific optimizer. It implements the batched pricing surface and returns an array-backed `DemandBatch`. `BundleFeatures` maps chosen bundles to feature rows; combRUM reuses the same feature map to build observed moments.

In [2]:
class BundleOracle(cb.Oracle):
    def __init__(
        self,
        X,
        shocks,
        bundles,
    ):
        self.X = np.asarray(X, dtype=np.float64)
        self.shocks = np.asarray(shocks, dtype=np.float64)
        self.bundles = np.asarray(bundles, dtype=np.float64)
        self.n_obs = int(self.X.shape[0])

    def price_batch(
        self,
        theta,
        local_ids,
    ):
        ids = np.asarray(local_ids, dtype=np.int64)
        i = ids % self.n_obs
        s = ids // self.n_obs
        utilities = np.einsum(
            "ijk,k->ij",
            self.X[i],
            np.asarray(theta, dtype=np.float64),
        )
        utilities = utilities + self.shocks[i, s]
        scores = np.einsum("ij,bj->ib", utilities, self.bundles)
        choices = np.argmax(scores, axis=1)
        payoffs = scores[np.arange(ids.size), choices]
        return cb.DemandBatch.exact(ids, self.bundles[choices], payoffs)


class BundleFeatures(cb.FeatureMap):
    def __init__(self, X, shocks):
        self.X = np.asarray(X, dtype=np.float64)
        self.shocks = np.asarray(shocks, dtype=np.float64)
        self.n_obs = int(self.X.shape[0])

    def features_batch(
        self,
        ids,
        bundles,
    ):
        ids = np.asarray(ids, dtype=np.int64)
        B = np.asarray(bundles, dtype=np.float64)
        i = ids % self.n_obs
        s = ids // self.n_obs
        Phi = np.einsum("ij,ijk->ik", B, self.X[i])
        Eps = np.einsum("ij,ij->i", B, self.shocks[i, s])
        return Phi, Eps

## Model And Data

`Parameters` names and bounds the flat theta vector. `Model` binds the oracle, parameter layout, feature map, and formulation. `Data` supplies observed bundles, simulation draws, and observed covariates.

In [3]:
parameters = cb.Parameters({"taste": (-2.0, 2.0, N_FEATURES)})
features = BundleFeatures(X, estimation_shocks)
oracle = BundleOracle(X, estimation_shocks, BUNDLES)
model = cb.Model(
    oracle,
    parameters,
    features=features,
    formulation=cb.NSlack,
)
data = cb.Data(
    observed_bundles=observed_bundles,
    shocks=estimation_shocks,
    observables=X,
)
theta_true = parameters.pack({"taste": THETA_TRUE})

{
    "parameters": repr(parameters),
    "blocks": parameters.names,
    "theta_true": parameters.unpack(theta_true),
    "formulation": model.formulation.__name__,
}

{'parameters': 'Parameters(K=3: taste[3])',
 'blocks': ('taste',),
 'theta_true': {'taste': array([ 0.9 , -0.5 ,  0.35])},
 'formulation': 'NSlack'}

## Point Estimate

`estimate` runs row generation. The result carries the estimate, named parameter blocks, convergence metadata, certification of pricing exactness, optional slack, and optional active cuts.

In [4]:
fit = cb.estimate(
    model,
    data,
    master_backend="highs",
    tolerance=1e-8,
    max_iterations=100,
    return_slack=True,
    return_cuts=True,
)

recovery_error = fit.theta_hat - theta_true

{
    "theta_true": THETA_TRUE.round(4).tolist(),
    "theta_hat": fit.theta_named()["taste"].round(4).tolist(),
    "recovery_error": recovery_error.round(4).tolist(),
    "max_abs_recovery_error": round(float(np.max(np.abs(recovery_error))), 4),
    "objective": round(float(fit.objective), 4),
    "converged": bool(fit.metadata["converged"]),
    "iterations": int(fit.metadata["iterations"]),
    "active_cuts": int(fit.n_active_cuts),
    "max_slack": round(fit.slack_summary()["max_slack"], 4),
}

{'theta_true': [0.9, -0.5, 0.35],
 'theta_hat': [0.8834, -0.4872, 0.3418],
 'recovery_error': [-0.0166, 0.0128, -0.0082],
 'max_abs_recovery_error': 0.0166,
 'objective': 4.0995,
 'converged': True,
 'iterations': 5,
 'active_cuts': 751,
 'max_slack': 4.8383}

## Bootstrap

`NativeDraws` provides reproducible multiplier weights. `BootstrapResult` summarizes all replications, with named accessors matching the parameter blocks.

In [5]:
boot = cb.bootstrap(
    model,
    data,
    n_bootstrap=10,
    weights=cb.NativeDraws(n_obs=N_OBS, base_seed=23),
    master_backend="highs",
    tolerance=1e-8,
    max_iterations=100,
)

ci_lo, ci_hi = boot.ci(only_converged=False)
{
    "n_converged": int(boot.n_converged),
    "mean": boot.mean(only_converged=False).round(4).tolist(),
    "se": boot.se_named(only_converged=False)["taste"].round(4).tolist(),
    "ci_95": [ci_lo.round(4).tolist(), ci_hi.round(4).tolist()],
}

{'n_converged': 10,
 'mean': [0.992, -0.5423, 0.3912],
 'se': [0.1439, 0.0716, 0.0606],
 'ci_95': [[0.775, -0.6194, 0.2955], [1.2267, -0.3995, 0.4855]]}

## Warm-Start A Follow-Up Fit

Warm starts are for follow-up fits where carried rows still mean the same thing. Here the second fit only tightens the tolerance on the same model and data, so the previous active cuts are valid seeds.

In [6]:
coarse = cb.estimate(
    model,
    data,
    master_backend="highs",
    tolerance=1e-4,
    max_iterations=30,
    return_cuts=True,
)
polished = cb.estimate(
    model,
    data,
    master_backend="highs",
    tolerance=1e-8,
    max_iterations=100,
    warm_start=coarse,
    warm_cuts=coarse.cuts,
    return_cuts=True,
)

{
    "coarse_iterations": int(coarse.metadata["iterations"]),
    "polished_iterations": int(polished.metadata["iterations"]),
    "polished_theta": polished.theta_named()["taste"].round(4).tolist(),
    "same_solution": bool(np.allclose(coarse.theta_hat, polished.theta_hat)),
}

{'coarse_iterations': 5,
 'polished_iterations': 1,
 'polished_theta': [0.8834, -0.4872, 0.3418],
 'same_solution': True}

## Next

- `notebooks/02_blp_bundle_demand.ipynb`: a multi-market bundle-demand application with a MIP demand oracle.
- `notebooks/03_network_formation.ipynb`: network formation with a min-cut pricing oracle.
- `notebooks/04_unitdemand_blp_large.ipynb`: market-wise `cb.OneSlack` estimation for large panels.
- `notebooks/05_peer_effects_large_network.ipynb`: large-network peer effects with a persistent NSlack master and MPI row generation.